In [ ]:
import os
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import uvicorn
import threading

In [ ]:
class EmbedRequest(BaseModel):
    texts: List[str]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
model.eval()

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(request: EmbedRequest):

    embeddings = model.encode(
        request.texts,
        batch_size=32,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    return {"embeddings": embeddings.cpu().tolist()}

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

INFO:     Started server process [5795]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8000

2026-02-22T18:22:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-22T18:22:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-22T18:22:41Z INF +--------------------------------------------------------------------------------------------+
2026-02-22T18:22:41Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-02-22T18:22:41Z INF |  https://raid-dem-meals-bolt.trycloudflare.com        

In [ ]:
!pkill -f uvicorn